Data Preparation

In [20]:
import pandas as pd
from pandas.api.types import is_numeric_dtype

df_vanilla = pd.read_csv("course_lead_scoring_2026.csv")

df_prepared = df_vanilla.copy()
for col in df_prepared.columns:
    if is_numeric_dtype(df_prepared[col]):
        filling_word = 0.0
    else:
        filling_word = "NA"
    df_prepared[col] = df_prepared[col].fillna(filling_word)

Answer of Question 1: technology

In [21]:
print(df_prepared["industry"].value_counts())

industry
technology       1173
retail            925
healthcare        840
finance           720
education         639
manufacturing     463
NA                240
Name: count, dtype: int64


Answer of Question 2: `interaction_count` and `lead_score`

In [22]:
corr_matrix = df_prepared.corr(numeric_only=True)
print(corr_matrix)

                          annual_income  number_of_courses_viewed  \
annual_income                  1.000000                  0.161300   
number_of_courses_viewed       0.161300                  1.000000   
interaction_count              0.122842                  0.721609   
lead_score                     0.229496                  0.757204   
converted                      0.189652                  0.395038   

                          interaction_count  lead_score  converted  
annual_income                      0.122842    0.229496   0.189652  
number_of_courses_viewed           0.721609    0.757204   0.395038  
interaction_count                  1.000000    0.915746   0.448624  
lead_score                         0.915746    1.000000   0.483246  
converted                          0.448624    0.483246   1.000000  


Split the data

In [23]:
from sklearn.model_selection import train_test_split
df_full_train, df_test = train_test_split(df_prepared, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=42
)

Answer of Question 3: `lead_source`

In [25]:
from sklearn.metrics import mutual_info_score

mis = {}
categorical = ['lead_source','industry','employment_status','location']
for col in df_train[categorical].columns:
    mis[col] = mutual_info_score(df_train[col],df_train['converted'])

print(mis)

{'lead_source': 0.03434605791617973, 'industry': 0.0019945636715935217, 'employment_status': 0.018965604906982875, 'location': 0.001042763579701525}


Answer of Question 4: 0.65

In [44]:
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression

train_dict = df_train.drop(columns=['converted']).to_dict(orient='records')
y_train = df_train['converted']
dv = DictVectorizer(sparse=False)
X_train = dv.fit_transform(train_dict)

model = LogisticRegression(
    solver='liblinear',
    C=1.0,
    max_iter=1000,
    random_state=42
)

model.fit(X_train, y_train)

w0 = model.intercept_[0]
w = model.coef_[0]

val_dict = df_val.drop(columns=['converted']).to_dict(orient='records')
y_val = df_val['converted']
X_val = dv.transform(val_dict)
y_pred = model.predict(X_val)
accuracy = (y_pred == y_val).mean()
print(f"accuracy: {accuracy:.10f}")
print(f"accuracy after rounded: {round(accuracy,2)}")


accuracy: 0.6450000000
accuracy after rounded: 0.64


Answer of Question 5: 'number_of_courses_viewed'

In [50]:
features = df_train.drop(columns=['converted']).columns
for col in features:
    train_dict = df_train.drop(columns=['converted',col]).to_dict(orient='records')
    y_train = df_train['converted']
    X_train = dv.fit_transform(train_dict)

    model.fit(X_train, y_train)

    w0 = model.intercept_[0]
    w = model.coef_[0]

    val_dict = df_val.drop(columns=['converted',col]).to_dict(orient='records')
    y_val = df_val['converted']
    X_val = dv.transform(val_dict)
    y_pred = model.predict(X_val)
    accuracy_tmp = (y_pred == y_val).mean()

    print(f"Difference after getting rid of {col}: {accuracy_tmp-accuracy}")

Difference after getting rid of lead_source: -0.0030000000000000027
Difference after getting rid of industry: 0.0
Difference after getting rid of employment_status: 0.0
Difference after getting rid of location: 0.0
Difference after getting rid of annual_income: 0.07899999999999996
Difference after getting rid of number_of_courses_viewed: -0.0020000000000000018
Difference after getting rid of interaction_count: -0.04400000000000004
Difference after getting rid of lead_score: -0.0010000000000000009


In [51]:
C = [0.000001, 0.00001, 0.0001, 0.001]

for c in C:
    train_dict = df_train.drop(columns=['converted']).to_dict(orient='records')
    y_train = df_train['converted']
    dv = DictVectorizer(sparse=False)
    X_train = dv.fit_transform(train_dict)

    model = LogisticRegression(
        solver='liblinear',
        C=c,
        max_iter=1000,
        random_state=42
    )

    model.fit(X_train, y_train)

    w0 = model.intercept_[0]
    w = model.coef_[0]

    val_dict = df_val.drop(columns=['converted']).to_dict(orient='records')
    y_val = df_val['converted']
    X_val = dv.transform(val_dict)
    y_pred = model.predict(X_val)
    accuracy = (y_pred == y_val).mean()
    print(f"Accuracy of using c={c}: {accuracy:.10f}")

Accuracy of using c=1e-06: 0.5980000000
Accuracy of using c=1e-05: 0.5980000000
Accuracy of using c=0.0001: 0.6130000000
Accuracy of using c=0.001: 0.6450000000
